# FinBERT Example Notebook

This notebooks shows how to train and use the FinBERT pre-trained language model for financial sentiment analysis.

## Modules 

In [1]:
from pathlib import Path
import shutil
import os
import logging
import sys
sys.path.append('..')

from textblob import TextBlob
from pprint import pprint
from sklearn.metrics import classification_report

from transformers import AutoModelForSequenceClassification

from finbert.finbert import *
import finbert.utils as tools

%load_ext autoreload
%autoreload 2

project_dir = Path.cwd().parent
pd.set_option('max_colwidth', None)

In [2]:
logging.basicConfig(format = '%(asctime)s - %(levelname)s - %(name)s -   %(message)s',
                    datefmt = '%m/%d/%Y %H:%M:%S',
                    level = logging.ERROR)

## Prepare the model

### Setting path variables:
1. `lm_path`: the path for the pre-trained language model (If vanilla Bert is used then no need to set this one).
2. `cl_path`: the path where the classification model is saved.
3. `cl_data_path`: the path of the directory that contains the data files of `train.csv`, `validation.csv`, `test.csv`.
---

In the initialization of `bertmodel`, we can either use the original pre-trained weights from Google by giving `bm = 'bert-base-uncased`, or our further pre-trained language model by `bm = lm_path`


---
All of the configurations with the model is controlled with the `config` variable. 

In [3]:
lm_path = project_dir/'models'/'language_model'/'finbertTRC2'
cl_path = project_dir/'models'/'classifier_model'/'finbert-sentiment'
cl_data_path = project_dir/'data'/'sentiment_data'

###  Configuring training parameters

You can find the explanations of the training parameters in the class docsctrings. 

In [4]:
# Clean the cl_path
try:
    shutil.rmtree(cl_path) 
except:
    pass

# Convert to absolute path string
import os
lm_path_str = os.path.abspath(str(lm_path))

# Check if the model path exists
if not os.path.exists(lm_path_str):
    print(f"Warning: Model path does not exist: {lm_path_str}")
    print("You can either:")
    print("1. Download the model and place it in the above path")
    print("2. Use 'bert-base-uncased' instead by setting: bertmodel = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)")
    print("\nTrying to use 'bert-base-uncased' as fallback...")
    bertmodel = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)
else:
    # Path exists, load from local directory
    # Note: For newer transformers versions, we need to ensure the path is recognized as local
    bertmodel = AutoModelForSequenceClassification.from_pretrained(lm_path_str, cache_dir=None, num_labels=3, local_files_only=True)


config = Config(   data_dir=cl_data_path,
                   bert_model=bertmodel,
                   num_train_epochs=4,
                   model_dir=cl_path,
                   max_seq_length = 48,
                   train_batch_size = 32,
                   learning_rate = 2e-5,
                   output_mode='classification',
                   warm_up_proportion=0.2,
                   local_rank=-1,
                   discriminate=True,
                   gradual_unfreeze=True)

You can either:
1. Download the model and place it in the above path
2. Use 'bert-base-uncased' instead by setting: bertmodel = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

Trying to use 'bert-base-uncased' as fallback...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


`finbert` is our main class that encapsulates all the functionality. The list of class labels should be given in the prepare_model method call with label_list parameter.

In [5]:
finbert = FinBert(config)
finbert.base_model = 'bert-base-uncased'
finbert.config.discriminate=True
finbert.config.gradual_unfreeze=True

In [6]:
finbert.prepare_model(label_list=['positive','negative','neutral'])

12/07/2025 11:25:28 - INFO - finbert.finbert -   device: mps n_gpu: 1, distributed training: False, 16-bits training: False


## Fine-tune the model

In [7]:
# Get the training examples
train_data = finbert.get_data('train')

In [8]:
model = finbert.create_the_model()

### [Optional] Fine-tune only a subset of the model
The variable `freeze` determines the last layer (out of 12) to be freezed. You can skip this part if you want to fine-tune the whole model.

<span style="color:red">Important: </span>
Execute this step if you want a shorter training time in the expense of accuracy.

In [9]:
# This is for fine-tuning a subset of the model.

freeze = 6

for param in model.bert.embeddings.parameters():
    param.requires_grad = False
    
for i in range(freeze):
    for param in model.bert.encoder.layer[i].parameters():
        param.requires_grad = False

### Training

In [10]:
trained_model = finbert.train(train_examples = train_data, model = model)

12/07/2025 11:25:29 - INFO - finbert.utils -   Label map: {'positive': 0, 'negative': 1, 'neutral': 2, None: 9090}
12/07/2025 11:25:29 - INFO - finbert.utils -   Label list: ['positive', 'negative', 'neutral']
12/07/2025 11:25:29 - INFO - finbert.utils -   *** Example ***
12/07/2025 11:25:29 - INFO - finbert.utils -   guid: train-1
12/07/2025 11:25:29 - INFO - finbert.utils -   tokens: [CLS] tel ##ias ##one ##ra s subsidiary , the mobile operator em ##t in estonia , has created a world s first mobile identification service which makes it possible to vote via a mobile phone . [SEP]
12/07/2025 11:25:29 - INFO - finbert.utils -   input_ids: 101 10093 7951 5643 2527 1055 7506 1010 1996 4684 6872 7861 2102 1999 10692 1010 2038 2580 1037 2088 1055 2034 4684 8720 2326 2029 3084 2009 2825 2000 3789 3081 1037 4684 3042 1012 102 0 0 0 0 0 0 0 0 0 0 0
12/07/2025 11:25:29 - INFO - finbert.utils -   attention_mask: 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 

Iteration:   0%|          | 0/122 [00:00<?, ?it/s]

12/07/2025 11:25:35 - INFO - finbert.utils -   *** Example ***
12/07/2025 11:25:35 - INFO - finbert.utils -   guid: validation-1
12/07/2025 11:25:35 - INFO - finbert.utils -   tokens: [CLS] ` ` my wife is looking forward to getting a pay ##che ##ck again , ' ' he qui ##pped recently as a six - knot current swirled around his anchored and heavily sponsored jet sl ##ed . [SEP]
12/07/2025 11:25:35 - INFO - finbert.utils -   input_ids: 101 1036 1036 2026 2564 2003 2559 2830 2000 2893 1037 3477 5403 3600 2153 1010 1005 1005 2002 21864 11469 3728 2004 1037 2416 1011 12226 2783 19171 2105 2010 14453 1998 4600 6485 6892 22889 2098 1012 102 0 0 0 0 0 0 0 0
12/07/2025 11:25:35 - INFO - finbert.utils -   attention_mask: 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0
12/07/2025 11:25:35 - INFO - finbert.utils -   token_type_ids: 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
12/07/2025 11:25:35 - INFO

Validating:   0%|          | 0/16 [00:00<?, ?it/s]

Validation losses: [0.9853280894458294]
No best model found


...

Iteration:   0%|          | 0/122 [00:00<?, ?it/s]

12/07/2025 11:26:12 - INFO - finbert.utils -   *** Example ***
12/07/2025 11:26:12 - INFO - finbert.utils -   guid: validation-1
12/07/2025 11:26:12 - INFO - finbert.utils -   tokens: [CLS] ` ` my wife is looking forward to getting a pay ##che ##ck again , ' ' he qui ##pped recently as a six - knot current swirled around his anchored and heavily sponsored jet sl ##ed . [SEP]
12/07/2025 11:26:12 - INFO - finbert.utils -   input_ids: 101 1036 1036 2026 2564 2003 2559 2830 2000 2893 1037 3477 5403 3600 2153 1010 1005 1005 2002 21864 11469 3728 2004 1037 2416 1011 12226 2783 19171 2105 2010 14453 1998 4600 6485 6892 22889 2098 1012 102 0 0 0 0 0 0 0 0
12/07/2025 11:26:12 - INFO - finbert.utils -   attention_mask: 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0
12/07/2025 11:26:12 - INFO - finbert.utils -   token_type_ids: 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
12/07/2025 11:26:12 - INFO

Validating:   0%|          | 0/16 [00:00<?, ?it/s]

Validation losses: [0.9853280894458294, 0.5591185819357634, 0.44629862532019615, 0.4345964565873146]


Epoch: 100%|██████████| 4/4 [00:44<00:00, 11.05s/it]


## Test the model

`bert.evaluate` outputs the DataFrame, where true labels and logit values for each example is given

In [11]:
test_data = finbert.get_data('test')

In [12]:
results = finbert.evaluate(examples=test_data, model=trained_model)

12/07/2025 11:26:15 - INFO - finbert.utils -   *** Example ***
12/07/2025 11:26:15 - INFO - finbert.utils -   guid: test-1
12/07/2025 11:26:15 - INFO - finbert.utils -   tokens: [CLS] finnish scan ##fi ##l , a systems supplier and contract manufacturer electronics industry , reports net sales of eu ##r 49 . 6 mn in the first quarter of 2009 , which are only a per cent smaller than in the corresponding period in 2008 . [SEP]
12/07/2025 11:26:15 - INFO - finbert.utils -   input_ids: 101 6983 13594 8873 2140 1010 1037 3001 17024 1998 3206 7751 8139 3068 1010 4311 5658 4341 1997 7327 2099 4749 1012 1020 24098 1999 1996 2034 4284 1997 2268 1010 2029 2024 2069 1037 2566 9358 3760 2084 1999 1996 7978 2558 1999 2263 1012 102
12/07/2025 11:26:15 - INFO - finbert.utils -   attention_mask: 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
12/07/2025 11:26:15 - INFO - finbert.utils -   token_type_ids: 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

Testing:   0%|          | 0/16 [00:00<?, ?it/s]

### Prepare the classification report

In [13]:
def report(df, cols=['label','prediction','logits']):
    #print('Validation loss:{0:.2f}'.format(metrics['best_validation_loss']))
    # Filter out rows with invalid labels (9090 is placeholder for None/missing labels)
    valid_mask = df[cols[0]] != 9090
    df_valid = df[valid_mask].copy()
    df_no_labels = df[~valid_mask].copy()
    
    # If we have valid labels, compute metrics
    if len(df_valid) > 0:
        cs = CrossEntropyLoss(weight=finbert.class_weights)
        loss = cs(torch.tensor(list(df_valid[cols[2]])),torch.tensor(list(df_valid[cols[0]])))
        print("Loss:{0:.2f}".format(loss))
        print("Accuracy:{0:.2f}".format((df_valid[cols[0]] == df_valid[cols[1]]).sum() / len(df_valid)) )
        print("\nClassification Report (with labels):")
        print(classification_report(df_valid[cols[0]], df_valid[cols[1]]))
    else:
        print("No ground truth labels available. Showing predictions only.")
    
    # Show prediction distribution
    if len(df) > 0:
        print("\nPrediction Distribution:")
        pred_counts = df[cols[1]].value_counts().sort_index()
        label_names = ['positive', 'negative', 'neutral']
        for idx, count in pred_counts.items():
            label_name = label_names[int(idx)] if int(idx) < len(label_names) else f'class_{idx}'
            print(f'  {label_name}: {count} ({count/len(df)*100:.1f}%)')


In [14]:
results['prediction'] = results.predictions.apply(lambda x: np.argmax(x,axis=0))

In [15]:
report(results,cols=['labels','prediction','predictions'])

No ground truth labels available. Showing predictions only.

Prediction Distribution:
  positive: 189 (39.0%)
  negative: 70 (14.4%)
  neutral: 226 (46.6%)


### Get predictions

With the `predict` function, given a piece of text, we split it into a list of sentences and then predict sentiment for each sentence. The output is written into a dataframe. Predictions are represented in three different columns: 

1) `logit`: probabilities for each class

2) `prediction`: predicted label

3) `sentiment_score`: sentiment score calculated as: probability of positive - probability of negative

Below we analyze a paragraph taken out of [this](https://www.economist.com/finance-and-economics/2019/01/03/a-profit-warning-from-apple-jolts-markets) article from The Economist. For comparison purposes, we also put the sentiments predicted with TextBlob.
> Later that day Apple said it was revising down its earnings expectations in the fourth quarter of 2018, largely because of lower sales and signs of economic weakness in China. The news rapidly infected financial markets. Apple’s share price fell by around 7% in after-hours trading and the decline was extended to more than 10% when the market opened. The dollar fell by 3.7% against the yen in a matter of minutes after the announcement, before rapidly recovering some ground. Asian stockmarkets closed down on January 3rd and European ones opened lower. Yields on government bonds fell as investors fled to the traditional haven in a market storm.

### Evaluate on Validation Set (with labels)

Evaluate the model on validation data which has ground truth labels to get metrics.

In [16]:
# Get validation data (which has labels)
validation_data = finbert.get_data("validation")

# Evaluate on validation set
val_results = finbert.evaluate(examples=validation_data, model=trained_model)
val_results["prediction"] = val_results.predictions.apply(lambda x: np.argmax(x,axis=0))

# Generate full report with metrics
report(val_results, cols=["labels","prediction","predictions"])

12/07/2025 11:26:15 - INFO - finbert.utils -   *** Example ***
12/07/2025 11:26:15 - INFO - finbert.utils -   guid: validation-1
12/07/2025 11:26:15 - INFO - finbert.utils -   tokens: [CLS] ` ` my wife is looking forward to getting a pay ##che ##ck again , ' ' he qui ##pped recently as a six - knot current swirled around his anchored and heavily sponsored jet sl ##ed . [SEP]
12/07/2025 11:26:15 - INFO - finbert.utils -   input_ids: 101 1036 1036 2026 2564 2003 2559 2830 2000 2893 1037 3477 5403 3600 2153 1010 1005 1005 2002 21864 11469 3728 2004 1037 2416 1011 12226 2783 19171 2105 2010 14453 1998 4600 6485 6892 22889 2098 1012 102 0 0 0 0 0 0 0 0
12/07/2025 11:26:15 - INFO - finbert.utils -   attention_mask: 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0
12/07/2025 11:26:15 - INFO - finbert.utils -   token_type_ids: 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
12/07/2025 11:26:15 - INFO

Testing:   0%|          | 0/16 [00:00<?, ?it/s]

Loss:0.44
Accuracy:0.79

Classification Report (with labels):
              precision    recall  f1-score   support

           0       0.69      0.79      0.74       136
           1       0.63      0.92      0.75        61
           2       0.91      0.76      0.83       288

    accuracy                           0.79       485
   macro avg       0.74      0.82      0.77       485
weighted avg       0.81      0.79      0.79       485


Prediction Distribution:
  positive: 155 (32.0%)
  negative: 89 (18.4%)
  neutral: 241 (49.7%)


/var/folders/6z/x30k7lfx3s14wgz7gd9kb3yh0000gn/T/ipykernel_51661/1786955391.py:11: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  loss = cs(torch.tensor(list(df_valid[cols[2]])),torch.tensor(list(df_valid[cols[0]])))


In [17]:
text = "Later that day Apple said it was revising down its earnings expectations in \
the fourth quarter of 2018, largely because of lower sales and signs of economic weakness in China. \
The news rapidly infected financial markets. Apple’s share price fell by around 7% in after-hours \
trading and the decline was extended to more than 10% when the market opened. The dollar fell \
by 3.7% against the yen in a matter of minutes after the announcement, before rapidly recovering \
some ground. Asian stockmarkets closed down on January 3rd and European ones opened lower. \
Yields on government bonds fell as investors fled to the traditional haven in a market storm."

In [18]:
cl_path = project_dir/'models'/'classifier_model'/'finbert-sentiment'
cl_path_str = os.path.abspath(str(cl_path))

# Check if the model path exists
if not os.path.exists(cl_path_str):
    raise FileNotFoundError(f"Classifier model path does not exist: {cl_path_str}. Please train the model first or ensure the path is correct.")
else:
    model = AutoModelForSequenceClassification.from_pretrained(cl_path_str, cache_dir=None, num_labels=3, local_files_only=True)

In [19]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /Users/bruno/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [20]:
result = predict(text,model)

12/07/2025 11:26:16 - INFO - root -   Using device: cpu 
12/07/2025 11:26:16 - INFO - finbert.utils -   *** Example ***
12/07/2025 11:26:16 - INFO - finbert.utils -   guid: 0
12/07/2025 11:26:16 - INFO - finbert.utils -   tokens: [CLS] later that day apple said it was rev ##ising down its earnings expectations in the fourth quarter of 2018 , largely because of lower sales and signs of economic weakness in china . [SEP]
12/07/2025 11:26:16 - INFO - finbert.utils -   input_ids: 101 2101 2008 2154 6207 2056 2009 2001 7065 9355 2091 2049 16565 10908 1999 1996 2959 4284 1997 2760 1010 4321 2138 1997 2896 4341 1998 5751 1997 3171 11251 1999 2859 1012 102 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
12/07/2025 11:26:16 - INFO - finbert.utils -   attention_mask: 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
12/07/2025 11:26:16 - INFO - finbert.utils -   token_type_ids: 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

In [21]:
blob = TextBlob(text)
result['textblob_prediction'] = [sentence.sentiment.polarity for sentence in blob.sentences]
result

,sentence,logit,prediction,sentiment_score,textblob_prediction
0,"Later that day Apple said it was revising down its earnings expectations in the fourth quarter of 2018, largely because of lower sales and signs of economic weakness in China.","[0.04243651, 0.9200903, 0.037473205]",negative,-0.877654,0.051746
1,The news rapidly infected financial markets.,"[0.04939766, 0.8197934, 0.13080898]",negative,-0.770396,0.000000
2,Apple’s share price fell by around 7% in after-hours trading and the decline was extended to more than 10% when the market opened.,"[0.040842324, 0.924358, 0.034799624]",negative,-0.883516,0.500000
3,"The dollar fell by 3.7% against the yen in a matter of minutes after the announcement, before rapidly recovering some ground.","[0.063213356, 0.9062258, 0.030560812]",negative,-0.843012,0.000000
4,Asian stockmarkets closed down on January 3rd and European ones opened lower.,"[0.034497675, 0.9157961, 0.0497062]",negative,-0.881298,-0.051111
5,Yields on government bonds fell as investors fled to the traditional haven in a market storm.,"[0.038770128, 0.9101999, 0.05102996]",negative,-0.871430,0.000000


In [22]:
print(f'Average sentiment is %.2f.' % (result.sentiment_score.mean()))

Average sentiment is -0.85.


Here is another example

In [23]:
text2 = "Shares in the spin-off of South African e-commerce group Naspers surged more than 25% \
in the first minutes of their market debut in Amsterdam on Wednesday. Bob van Dijk, CEO of \
Naspers and Prosus Group poses at Amsterdam's stock exchange, as Prosus begins trading on the \
Euronext stock exchange in Amsterdam, Netherlands, September 11, 2019. REUTERS/Piroschka van de Wouw \
Prosus comprises Naspers’ global empire of consumer internet assets, with the jewel in the crown a \
31% stake in Chinese tech titan Tencent. There is 'way more demand than is even available, so that’s \
good,' said the CEO of Euronext Amsterdam, Maurice van Tilburg. 'It’s going to be an interesting \
hour of trade after opening this morning.' Euronext had given an indicative price of 58.70 euros \
per share for Prosus, implying a market value of 95.3 billion euros ($105 billion). The shares \
jumped to 76 euros on opening and were trading at 75 euros at 0719 GMT."

In [24]:
result2 = predict(text2,model)
blob = TextBlob(text2)
result2['textblob_prediction'] = [sentence.sentiment.polarity for sentence in blob.sentences]

12/07/2025 11:26:17 - INFO - root -   Using device: cpu 
12/07/2025 11:26:17 - INFO - finbert.utils -   *** Example ***
12/07/2025 11:26:17 - INFO - finbert.utils -   guid: 0
12/07/2025 11:26:17 - INFO - finbert.utils -   tokens: [CLS] shares in the spin - off of south african e - commerce group nas ##pers surged more than 25 % in the first minutes of their market debut in amsterdam on wednesday . [SEP]
12/07/2025 11:26:17 - INFO - finbert.utils -   input_ids: 101 6661 1999 1996 6714 1011 2125 1997 2148 3060 1041 1011 6236 2177 17235 7347 18852 2062 2084 2423 1003 1999 1996 2034 2781 1997 2037 3006 2834 1999 7598 2006 9317 1012 102 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
12/07/2025 11:26:17 - INFO - finbert.utils -   attention_mask: 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
12/07/2025 11:26:17 - INFO - finbert.utils -   token_type_ids: 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

In [25]:
result2

,sentence,logit,prediction,sentiment_score,textblob_prediction
0,Shares in the spin-off of South African e-commerce group Naspers surged more than 25% in the first minutes of their market debut in Amsterdam on Wednesday.,"[0.88324773, 0.031404044, 0.08534822]",positive,0.851844,0.250000
1,"Bob van Dijk, CEO of Naspers and Prosus Group poses at Amsterdam's stock exchange, as Prosus begins trading on the Euronext stock exchange in Amsterdam, Netherlands, September 11, 2019.","[0.21001336, 0.037846956, 0.7521397]",neutral,0.172166,0.000000
2,"REUTERS/Piroschka van de Wouw Prosus comprises Naspers’ global empire of consumer internet assets, with the jewel in the crown a 31% stake in Chinese tech titan Tencent.","[0.15978119, 0.028237833, 0.81198096]",neutral,0.131543,0.000000
3,"There is 'way more demand than is even available, so that’s good,' said the CEO of Euronext Amsterdam, Maurice van Tilburg.","[0.8133972, 0.032449216, 0.15415362]",positive,0.780948,0.533333
4,'It’s going to be an interesting hour of trade after opening this morning.',"[0.50365806, 0.069383405, 0.4269585]",positive,0.434275,0.500000
5,"Euronext had given an indicative price of 58.70 euros per share for Prosus, implying a market value of 95.3 billion euros ($105 billion).","[0.19618423, 0.036309388, 0.76750636]",neutral,0.159875,0.000000
6,The shares jumped to 76 euros on opening and were trading at 75 euros at 0719 GMT.,"[0.6799258, 0.029093705, 0.29098046]",positive,0.650832,0.000000


In [26]:
print(f'Average sentiment is %.2f.' % (result2.sentiment_score.mean()))

Average sentiment is 0.45.
